# ⚡ ADAPT ATC — Quick Training (30 min, T4/A100)

**Standalone notebook — no repo imports needed. Just run all cells.**

| | |
|--|--|
| Time | ~30 min on T4 |
| Model | Qwen2.5-1.5B-Instruct 4-bit |
| Tasks | Mini ATC (3–5 flights, 1 runway) |
| Reward | Progressive: format→coverage→priority |
| Goal | See reward rise from ~0.10 → ~0.65 |

Works on **T4 (fp16)** and **A100 (bf16)** — auto-detects.

> After this quick run succeeds, open `train_t4.ipynb` or `train_a100.ipynb` for full training with the complete ADAPT dataset.


In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────────
import subprocess, sys, os

def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)

pip("unsloth")
pip("trl>=0.15", "datasets", "bitsandbytes", "accelerate", "peft", "matplotlib")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ Done")

In [ ]:
# ── 2. GPU + dtype ────────────────────────────────────────────────────────────
import torch

assert torch.cuda.is_available(), "❌ Enable GPU: Runtime → Change runtime type → GPU"

gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_BF16 = torch.cuda.is_bf16_supported()   # True on A100, False on T4

print(f"GPU    : {gpu}")
print(f"VRAM   : {vram:.1f} GB")
print(f"dtype  : {'bfloat16 (A100)' if IS_BF16 else 'float16 (T4)'}")
print()
print("T4  → fp16 only (Turing, no native bf16 matmul)")
print("A100→ bf16 only (Ampere Tensor Cores)")

In [ ]:
# ── 3. Load model ─────────────────────────────────────────────────────────────
from unsloth import FastLanguageModel, PatchFastRL

try:
    PatchFastRL("GRPO", FastLanguageModel)
    print("✅ PatchFastRL applied")
except Exception as e:
    print(f"ℹ️  PatchFastRL: {e}")

MAX_SEQ_LEN = 1024   # short prompts → fits in 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,           # auto: fp16 on T4, bf16 on A100
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✅ {sum(p.numel() for p in model.parameters())/1e6:.0f}M params | dtype={next(model.parameters()).dtype}")

In [ ]:
# ── 4. LoRA ───────────────────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
to = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA: {tr:,} / {to:,} trainable ({100*tr/to:.2f}%)")

In [ ]:
# ── 5. Mini ATC tasks (self-contained, no repo needed) ───────────────────────
#
# 24 micro-tasks: 3–5 flights, 1 runway, always ≥1 emergency.
# Short prompts (<250 tokens) → completions converge to valid JSON fast.

import random, copy

SYSTEM = """You are an Air Traffic Controller. Schedule arrivals on the given runway.

OUTPUT (strict JSON, no markdown, no extra text):
{"arrival_slots": [{"flight_id": "XXX", "runway": "RRR", "assigned_minute": N, "hold_minutes": N}], "rationale": "brief reason"}

RULES:
1. EMERGENCY arrives first — delay ≤5 min from scheduled.
2. Wake separation: H→H 4min, H→M 5min, H→L 6min, M→M 3min, M→L 4min, L→L 3min.
3. Each flight within [earliest, latest] window.
4. Assign ALL flights."""


def _make_flight(fid, priority, wake, sched, window, rwy):
    e, l = sched - window, sched + window
    return {"id": fid, "priority": priority, "wake": wake,
            "scheduled": sched, "earliest": e, "latest": l, "runway": rwy}


# Base templates — varied scenarios
_BASES = [
    # (flights_list, runway)
    ([_make_flight("EMR001","emergency","H",10,5,"28R"),
      _make_flight("AI101",  "normal",   "M",18,7,"28R"),
      _make_flight("6E202",  "normal",   "L",26,8,"28R")], "28R"),

    ([_make_flight("MED002","medical",   "M",12,4,"09L"),
      _make_flight("AI305", "normal",    "H",20,8,"09L"),
      _make_flight("SG401", "normal",    "M",28,8,"09L"),
      _make_flight("UK901", "connection","L",35,7,"09L")], "09L"),

    ([_make_flight("EMR003","emergency","L",8, 4,"14R"),
      _make_flight("6E501", "normal",   "M",15,6,"14R"),
      _make_flight("AI702", "normal",   "H",24,8,"14R"),
      _make_flight("GO121", "normal",   "L",33,8,"14R"),
      _make_flight("IX221", "normal",   "M",42,9,"14R")], "14R"),

    ([_make_flight("MED004","medical",   "H",15,5,"32L"),
      _make_flight("AI803", "connection","M",22,7,"32L"),
      _make_flight("UK601", "normal",    "L",30,8,"32L")], "32L"),

    ([_make_flight("EMR005","emergency","M",5, 3,"28R"),
      _make_flight("AI201", "normal",   "H",14,7,"28R"),
      _make_flight("6E301", "normal",   "M",22,8,"28R"),
      _make_flight("SG101", "normal",   "L",30,8,"28R")], "28R"),

    ([_make_flight("EMR006","emergency","H",20,5,"09L"),
      _make_flight("AI401", "normal",   "M",30,8,"09L"),
      _make_flight("GO201", "normal",   "L",40,9,"09L")], "09L"),
]


def _flight_to_str(f):
    return (f"  {f['id']:8s} {f['priority']:10s} {f['wake']} "
            f"sched={f['scheduled']:3d}min [{f['earliest']}-{f['latest']}]")


def _make_sample(base_idx, jitter=0, rng=None):
    if rng is None: rng = random
    flights, rwy = copy.deepcopy(_BASES[base_idx % len(_BASES)])
    # Small time jitter for diversity
    for f in flights:
        delta = rng.randint(-jitter, jitter) if jitter else 0
        f["scheduled"] = max(1, f["scheduled"] + delta)
        f["earliest"]  = max(0, f["earliest"]  + delta)
        f["latest"]    = f["latest"]   + delta

    # Shuffle non-emergency flights (emergency stays in list but prompt doesn't sort)
    emg  = [f for f in flights if f["priority"] in ("emergency", "medical")]
    rest = [f for f in flights if f not in emg]
    rng.shuffle(rest)
    prompt_flights = rest + emg   # emergency last in prompt = harder to trivially put first

    user = (
        f"Runway: {rwy}\n"
        f"Schedule these {len(flights)} arrivals:\n"
        + "\n".join(_flight_to_str(f) for f in prompt_flights)
        + "\n\nOutput JSON schedule."
    )
    import json
    return {
        "prompt":    [{"role": "system", "content": SYSTEM},
                      {"role": "user",   "content": user}],
        "task_id":   f"mini_{base_idx}_{jitter}",
        "flights_json": json.dumps(flights),
    }


rng = random.Random(42)
samples = []
for epoch in range(8):          # 8 epochs × 6 bases × jitter variants
    for bi in range(len(_BASES)):
        for jitter in (0, 2, 4):
            samples.append(_make_sample(bi, jitter=jitter, rng=rng))

rng.shuffle(samples)
samples = samples[:120]   # cap at 120 = ~25–30 min on T4

from datasets import Dataset
dataset = Dataset.from_list(samples)
print(f"✅ Dataset: {len(dataset)} mini-ATC samples")
print(f"   Sample prompt (user turn):\n")
print(samples[0]["prompt"][1]["content"])

In [ ]:
# ── 6. Progressive reward (4 tiers) ──────────────────────────────────────────
#
# Tier 1 FORMAT  (0–0.20): partial credit for any JSON-like text
# Tier 2 VALID   (0.20):   bonus for parseable JSON
# Tier 3 COVERAGE(0–0.30): fraction of flights assigned
# Tier 4 PRIORITY(0–0.30): emergency/medical lands first
# Tier 5 WINDOW  (0–0.15): flights within [earliest, latest]
# Tier 6 RATIONALE(0–0.05): non-empty explanation
#
# Expected cold-start distribution: 0.05–0.20
# Expected after 30 min:            0.50–0.75

import json as _json, re as _re


def _strip_md(text):
    return _re.sub(r"```(?:json)?\s*|\s*```", "", str(text)).strip()


def reward_fn(completions, **kwargs):
    n            = len(completions)
    flights_list = kwargs.get("flights_json", ["[]"] * n)
    if not isinstance(flights_list, list):
        flights_list = [flights_list] * n
    while len(flights_list) < n:
        flights_list.append(flights_list[-1] if flights_list else "[]")

    rewards = []
    for comp, fj in zip(completions, flights_list):
        if isinstance(comp, list):
            comp = comp[-1].get("content", "") if comp else ""
        text = str(comp)

        try:
            flights = _json.loads(fj)
        except Exception:
            flights = []

        # ── Tier 1: FORMAT — always non-zero for JSON-like content ────────
        score = 0.02   # baseline (all completions get ≥0.02 → reward_std > 0)
        if "{" in text:               score += 0.04
        if "arrival_slots" in text:   score += 0.06
        if '"flight_id"' in text:     score += 0.04
        if '"assigned_minute"' in text: score += 0.04

        # ── Tier 2: VALID JSON ────────────────────────────────────────────
        try:
            data  = _json.loads(_strip_md(text))
            slots = data.get("arrival_slots", [])
            if not isinstance(slots, list): slots = []
            slot_map = {s.get("flight_id"): s for s in slots
                        if isinstance(s, dict) and "flight_id" in s}
            score = max(score, 0.20)   # valid JSON floor

            fids = [f["id"] for f in flights]
            emg  = [f["id"] for f in flights
                    if f.get("priority") in ("emergency", "medical")]

            # ── Tier 3: COVERAGE ─────────────────────────────────────────
            covered = sum(1 for fid in fids if fid in slot_map)
            coverage_score = covered / max(1, len(fids))
            score += 0.30 * coverage_score

            # ── Tier 4: PRIORITY — emergency/medical must be assigned first ──
            if emg and slots:
                # Sort by assigned_minute to find actual first slot
                ordered = sorted(slots, key=lambda s: s.get("assigned_minute", 9999))
                first_id = ordered[0].get("flight_id", "") if ordered else ""
                if first_id in emg:
                    score += 0.30           # emergency is genuinely first
                elif any(s.get("flight_id") in emg
                         for s in ordered[:2]):
                    score += 0.12           # emergency in top-2: partial credit
            elif not emg:
                score += 0.30               # no emergency: full priority credit

            # ── Tier 5: WINDOW feasibility ────────────────────────────────
            in_window = 0
            for f in flights:
                s = slot_map.get(f["id"])
                if s:
                    t = s.get("assigned_minute", -1)
                    if f.get("earliest", 0) <= t <= f.get("latest", 9999):
                        in_window += 1
            if flights:
                score += 0.15 * (in_window / len(flights))

            # ── Tier 6: RATIONALE ─────────────────────────────────────────
            rat = str(data.get("rationale", "")).strip()
            if len(rat) > 10: score += 0.03
            if any(w in rat.lower() for w in
                   ("emergency", "priority", "wake", "first", "separation")):
                score += 0.02

        except (_json.JSONDecodeError, Exception):
            pass   # keep tier-1 partial credit

        rewards.append(float(min(score, 1.0)))

    return rewards


# ── Quick sanity check ────────────────────────────────────────────────────────
import json as _j
_flights = _j.dumps([{"id":"EMR001","priority":"emergency","wake":"H",
                       "scheduled":10,"earliest":5,"latest":15,"runway":"28R"},
                      {"id":"AI101", "priority":"normal",   "wake":"M",
                       "scheduled":18,"earliest":13,"latest":25,"runway":"28R"}])

_t_bad  = "{"                        # garbage
_t_ok   = '{"arrival_slots":[{"flight_id":"EMR001","runway":"28R","assigned_minute":10,"hold_minutes":0},{"flight_id":"AI101","runway":"28R","assigned_minute":15,"hold_minutes":0}],"rationale":"emergency first then normal with wake gap"}'
_t_poor = '{"arrival_slots":[{"flight_id":"AI101","runway":"28R","assigned_minute":10,"hold_minutes":0},{"flight_id":"EMR001","runway":"28R","assigned_minute":15,"hold_minutes":5}],"rationale":"scheduled order"}'

r_bad, r_poor, r_ok = reward_fn(
    [_t_bad, _t_poor, _t_ok],
    flights_json=[_flights, _flights, _flights],
)
print(f"✅ Reward sanity:")
print(f"   garbage (no JSON)        : {r_bad:.3f}  (format only)")
print(f"   valid JSON, wrong order  : {r_poor:.3f}  (coverage ok, priority wrong)")
print(f"   valid JSON, correct order: {r_ok:.3f}   (all tiers satisfied)")
assert r_bad < r_poor < r_ok, "Reward ordering broken!"
print("   Ordering correct: bad < wrong_order < correct ✅")

In [ ]:
# ── 7. GRPO config ────────────────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "./adapt-quick-out"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = GRPOConfig(
    # Generation
    num_generations=2,          # 2 = minimum for GRPO variance
    max_completion_length=128,  # short tasks = short completions
    max_prompt_length=512,      # mini tasks fit in 512

    # Optimisation
    learning_rate=8e-6,         # slightly higher LR for fast convergence
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=1,         # 1 epoch over 120 samples = fast
    warmup_ratio=0.05,
    max_grad_norm=1.0,

    # Precision — auto T4/A100
    bf16=IS_BF16,
    fp16=not IS_BF16,

    # Memory
    optim="adamw_8bit",
    gradient_checkpointing=True,

    # KL
    beta=0.0,

    # Logging
    output_dir=OUTPUT_DIR,
    logging_steps=1,
    save_steps=60,
    save_total_limit=1,
    report_to="none",
    seed=42,
    run_name="adapt-atc-quick",
)

print(f"✅ Config ready")
print(f"   dtype      : {'bf16' if IS_BF16 else 'fp16'}")
print(f"   samples    : {len(dataset)}")
print(f"   generations: {training_args.num_generations}")
print(f"   completion : {training_args.max_completion_length} tokens")

In [ ]:
# ── 8. Train ──────────────────────────────────────────────────────────────────
import time

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=dataset,
)

print("🚀 Quick training started!")
print("   Watch reward_mean in the log — should rise from ~0.10 to ~0.50+ over the run.")
print("   Expected: ~25–35 min on T4 for 120 samples\n")

t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"\n✅ Done in {elapsed/60:.0f} min")

In [ ]:
# ── 9. Reward curve ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

log = trainer.state.log_history

vals = []
for entry in log:
    for k in ["rewards/reward_fn", "reward", "train/reward",
              "rewards/combined_reward_fn", "rewards/reward_fn_mean"]:
        if k in entry and isinstance(entry[k], (int, float)):
            vals.append(float(entry[k]))
            break
if not vals:
    for entry in log:
        for k, v in entry.items():
            if "reward" in k.lower() and isinstance(v, (int, float)):
                vals.append(float(v)); break

if vals:
    W  = max(3, len(vals) // 10)
    ma = [sum(vals[max(0,i-W):i+1])/min(i+1,W) for i in range(len(vals))]

    plt.figure(figsize=(12, 4))
    plt.plot(vals, alpha=0.35, color="steelblue", label="step")
    plt.plot(ma, color="crimson", lw=2.5, label=f"MA-{W}")
    plt.axhline(0.5, ls="--", color="green", alpha=0.5, label="target 0.5")
    plt.xlabel("Step"); plt.ylabel("Reward")
    plt.title("Quick Training — Reward Curve")
    plt.legend(); plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/reward_quick.png", dpi=150)
    plt.show()

    q = max(1, len(vals)//4)
    f_, l_ = sum(vals[:q])/q, sum(vals[-q:])/q
    print(f"\n📊 First 25%: {f_:.4f}  |  Last 25%: {l_:.4f}  |  Δ {l_-f_:+.4f}")
    if l_ > f_ + 0.05:
        print("   ✅ Reward increasing — model is learning!")
        print("   Next step: train_t4.ipynb / train_a100.ipynb for full ADAPT dataset.")
    else:
        print("   ⚠️  Reward not clearly rising yet.")
        print("   Try: increase num_train_epochs to 3 in the config cell and re-run.")
else:
    print("No reward values in log. Keys:", list(log[0].keys()) if log else "empty")

In [ ]:
# ── 10. Save ──────────────────────────────────────────────────────────────────
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Saved to {OUTPUT_DIR}")

In [ ]:
# ── 11. Inference demo ────────────────────────────────────────────────────────
import torch, json, re

model.eval()

test_user = """Runway: 28R
Schedule these 4 arrivals:
  AI301    normal     M sched= 15min [10-22]
  EMR007   emergency  H sched= 10min [ 7-15]
  6E601    normal     L sched= 25min [20-33]
  GO301    connection M sched= 35min [30-42]

Output JSON schedule."""

messages = [{"role": "system", "content": SYSTEM},
            {"role": "user",   "content": test_user}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=128, temperature=0.3,
        do_sample=True, pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=" * 55)
print("Model output:")
print(response)
print("=" * 55)

try:
    data  = json.loads(re.sub(r"```(?:json)?\s*|\s*```", "", response).strip())
    slots = data.get("arrival_slots", [])
    ordered = sorted(slots, key=lambda s: s.get("assigned_minute", 9999))
    print(f"\n✅ Valid JSON — {len(slots)} slots")
    for s in ordered:
        tag = " ← CORRECT" if s.get("flight_id") == "EMR007" and ordered and ordered[0].get("flight_id") == "EMR007" and s == ordered[0] else ""
        print(f"   min {s.get('assigned_minute',0):3d}  {s.get('flight_id','?')}{tag}")
    first = ordered[0].get("flight_id","") if ordered else ""
    if first == "EMR007":
        print("\n🎯 Emergency EMR007 landed first — reward rule satisfied!")
    else:
        print(f"\nℹ️  First slot: {first} (EMR007 should be first — training will fix this)")
except json.JSONDecodeError:
    print("ℹ️  JSON not yet valid — normal early in training.")
    print("    More episodes will fix this. Run train_t4.ipynb for full training.")